# 11. 매도 ML·DL 공통 시작 코드 — 점검 (11/26)

대상: 팀원 4명 (각자 매도 ML·DL 을 돌리기 전에, 공통 시작 코드가 엔진과 같은 계산을 하는지 확인한 기록)

- **성격:** 팀원이 같이 쓰는 시작 코드 `indexbeater/sell_common.py` 점검. **이 노트북에서는 모델 학습·점수 매기기를 하지 않는다.**
  학습(`--run`)·고정 H 비교(`--baseline`)는 팀장 승인과 `실험기록.csv` 규칙 줄을 먼저 적은 뒤에 한다.
- 2020년 이후 자료는 열지 않는다 (데이터는 2019-12-31 까지만 읽음).
- 명령: `python -m indexbeater sellml --check` (점검, 저장 안 함) · `--list` (모델·설정·입력 목록).

**11/26 팀장 결정 (매도 부분만 바뀜)**
- 시장마다 모델 하나 (전략 이름은 입력에 안 넣음). 2015년 신호부터 학습. 정화 40거래일.
- **최대 보유 기간 없음.** 해가 바뀌면 그해 모델이 판단. 2019-12-30 까지 못 팔면 평가만 (미완료).
- 고정 손절·익절 없음 (고정 H5·H10·H20 은 비교 기준으로만).
- 라벨: d일 장 끝에 판단. 'd 다음 팔 수 있는 첫 시가' 와 'd+20 다음 팔 수 있는 첫 시가' 를 비교해 들고 있는 쪽이 0.5% 넘게 나쁘면 판다(1).
- 가중치: |차이| ÷ 그 매매의 줄 수. 학습 줄 = 매수 뒤 120거래일까지.
- 기준 확률 0.55 / 0.60 / 0.65 중 학습 구간 마지막 20% 에서 평균 순수익이 가장 큰 것.
- 점수: TOP3 6칸마다 포트폴리오, 첫째 순위 기준 = 지수 초과수익 중앙값.

1. 테스트 (가짜 데이터)
2. 실제 데이터 점검 (`--check`)
3. 모델·설정·입력 목록 (`--list`)
4. 정리: 확인한 것 / 확인 못한 것 / 이상한 것

이 노트북은 데이터 파일을 저장하지 않는다.

In [1]:
# 이 칸을 맨 먼저 실행한다.
# 노트북이 어느 폴더에서 열려도, indexbeater 폴더가 있는 프로젝트 폴더를 찾아서 길 목록(sys.path)에 넣는다.
import os
import sys

root = os.getcwd()
while not os.path.isdir(os.path.join(root, "indexbeater")) and os.path.dirname(root) != root:
    root = os.path.dirname(root)          # 못 찾으면 한 칸 위 폴더로 올라가서 다시 찾는다
sys.path.insert(0, root)
os.chdir(root)
print("프로젝트 폴더:", root)

프로젝트 폴더: c:\Users\gram\Desktop\project\buysignal


## 1. 테스트

`tests/check_period.py` 중 매도 ML 테스트 9개만 돌린다 (기본 pytest 130개 모음에는 넣지 않음).
- `test_sellml_fixed_and_portfolio_same_as_engine`: 고정 H5·H10·H20 매매 표 = 엔진, 포트폴리오 = 엔진 run_portfolio
- `test_sellml_settle_rule`: '판다' 한 다음 날부터 팔 수 있는 첫 시가 (거래정지 건너뜀) · 상장폐지 = 마지막 종가 · 끝까지 못 팔면 미완료
- `test_sellml_label_purge`: 라벨 계산 · 라벨 끝 날 · 정화 40 (학습 줄의 라벨 끝 + 20 < 시험 해 첫날, 앞 80% 와 검증 20% 도 같은 규칙)
- `test_sellml_no_lookahead`: 뒷날 가격을 마구 바꿔도 앞날 특징·학습 줄·그해 학습 결과는 그대로 (미래 엿보기 없음)
- `test_sellml_long_hold_and_chunks`: 최대 보유 한도 없음 (안 파는 모델은 끝까지 들고 감, 60일 묶음 넘어서도 판단)
- `test_sellml_weights_and_threshold`: 가중치 규칙 · 기준 확률 고르기 (같으면 앞 것)
- `test_sellml_models_fit_predict`: 모델 7종이 같은 틀로 학습·확률 · 설정 수 = 계획서 표 · 저장소에 파일 안 만듦
- `test_sellml_csv_checks`: 신호 CSV 가 내 데이터와 안 맞으면 멈춤
- `test_sellml_cell_portfolio`: 칸 포트폴리오 · 해마다 수익을 곱하면 마지막 평가금액

맨 아래 줄이 `passed` 이고 `failed` 가 없으면 통과.

In [2]:
import subprocess

r = subprocess.run([sys.executable, "-m", "pytest", "tests/check_period.py", "-k", "sellml", "--color=no", "-q",
                    "-o", "addopts="], capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-1500:] or r.stderr[-1500:])

.........                                                                [100%]
9 passed, 61 deselected in 22.52s



## 2. 실제 데이터 점검 (`--check`)

팀원도 학습 전에 **반드시 이것부터** 돌린다. 전부 ✓ 가 아니면 학습하지 말고 팀장에게 알린다.
필요한 파일: `results/top3_buy_signals.csv`, `results/team_compare_portfolio_daily.csv`, `results/benchmark_market.csv`.

- CSV 매수가·갭취소 = 내 데이터로 다시 계산한 값
- 고정 H5·H10·H20 매매 표 = 엔진 `simulate_trades` (줄 수·날짜·상태·가격 전부)
- H5·비용 0.5%·2015 신호부터 포트폴리오 = 11/26 회의 PDF 에 쓴 저장 평가금액 (6칸)
- 학습 줄 수: 판단한 날의 해마다 · 시험 해마다 (앞 80% / 검증 20% / 전부)

In [3]:
import time

t0 = time.time()
r = subprocess.run([sys.executable, "-m", "indexbeater", "sellml", "--check"], capture_output=True, text=True,
                   encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(r.stdout[-8000:] or r.stderr[-3000:])
print("끝난 코드:", r.returncode, "(0 이면 통과) · 걸린 시간:", round(time.time() - t0), "초")

  신호 30,837줄 (2015년부터) · 칸 6개
  데이터 1,998일 × 2,353종목 (9초)
  ✓ CSV 매수가 = T+1 수정 시가
  ✓ CSV 갭취소 = 내 계산
  ✓ 고정 H5 매매 표 = 엔진 (KOSPI, 12,323건)
  ✓ 고정 H5 매매 표 = 엔진 (KOSDAQ, 13,885건)
  ✓ 고정 H10 매매 표 = 엔진 (KOSPI, 12,323건)
  ✓ 고정 H10 매매 표 = 엔진 (KOSDAQ, 13,885건)
  ✓ 고정 H20 매매 표 = 엔진 (KOSPI, 12,323건)
  ✓ 고정 H20 매매 표 = 엔진 (KOSDAQ, 13,885건)
  ✓ H5·0.5% 포트폴리오 = PDF (KOSPI 나, 차이 2.2e-16)
  ✓ H5·0.5% 포트폴리오 = PDF (KOSPI 다-2, 차이 2.2e-16)
  ✓ H5·0.5% 포트폴리오 = PDF (KOSPI 가, 차이 2.2e-16)
  ✓ H5·0.5% 포트폴리오 = PDF (KOSDAQ 다-2, 차이 2.2e-16)
  ✓ H5·0.5% 포트폴리오 = PDF (KOSDAQ 나, 차이 2.2e-16)
  ✓ H5·0.5% 포트폴리오 = PDF (KOSDAQ 가, 차이 2.2e-16)

[점검]
                                          점검   통과
                         CSV 매수가 = T+1 수정 시가 True
                              CSV 갭취소 = 내 계산 True
            고정 H5 매매 표 = 엔진 (KOSPI, 12,323건) True
           고정 H5 매매 표 = 엔진 (KOSDAQ, 13,885건) True
           고정 H10 매매 표 = 엔진 (KOSPI, 12,323건) True
          고정 H10 매매 표 = 엔진 (KOSDAQ, 13,885건) True
           고정 H20 매매 표 = 엔진 (KO

## 3. 모델·설정·입력 목록 (`--list`)

계획서 5종 (설정 수 그대로) + 11/26 에 더한 XGBoost·CatBoost 5개씩. **설정 하나 = 시도 하나** 로 센다.
다른 모델은 11/27 전에 정해 두면 `sell_common.add_model(이름, 만드는함수, 설정목록)` 으로 넣을 수 있다.

In [4]:
r = subprocess.run([sys.executable, "-m", "indexbeater", "sellml", "--list"], capture_output=True, text=True,
                   encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(r.stdout or r.stderr)

  logistic       3개: [{'C': 0.01}, {'C': 0.1}, {'C': 1.0}]
  random_forest  4개: [{'max_depth': 6, 'min_samples_leaf': 500}, {'max_depth': 6, 'min_samples_leaf': 2000}, {'max_depth': 10, 'min_samples_leaf': 500}, {'max_depth': 10, 'min_samples_leaf': 2000}]
  lightgbm       5개: [{'num_leaves': 15, 'learning_rate': 0.05}, {'num_leaves': 31, 'learning_rate': 0.05}, {'num_leaves': 63, 'learning_rate': 0.05}, {'num_leaves': 31, 'learning_rate': 0.02}, {'num_leaves': 31, 'learning_rate': 0.1}]
  xgboost        5개: [{'max_depth': 3, 'learning_rate': 0.05}, {'max_depth': 4, 'learning_rate': 0.05}, {'max_depth': 6, 'learning_rate': 0.05}, {'max_depth': 4, 'learning_rate': 0.02}, {'max_depth': 4, 'learning_rate': 0.1}]
  catboost       5개: [{'depth': 4, 'learning_rate': 0.05}, {'depth': 6, 'learning_rate': 0.05}, {'depth': 8, 'learning_rate': 0.05}, {'depth': 6, 'learning_rate': 0.02}, {'depth': 6, 'learning_rate': 0.1}]
  lstm           3개: [{'units': 16}, {'units': 32}, {'units': 64}]
  cnn1d 

## 4. 정리

**확인한 것**
- 가짜 데이터 테스트 9개 통과 (위 1절). 기본 pytest 130개도 그대로 통과.
- 실제 데이터: CSV 매수가·갭취소, 고정 H5·H10·H20 매매 표 (KOSPI 12,323건 · KOSDAQ 13,885건) 가 엔진과 같다.
- 실제 데이터: H5·0.5% 포트폴리오 6칸이 회의 PDF 의 저장 평가금액과 같다 (차이 2.2e-16 = 컴퓨터 계산 끝자리).
- 학습 줄의 라벨 끝 날이 모두 시험 해 첫날보다 20거래일 넘게 앞이다 (위 표의 '가장늦은라벨끝' 이 해마다 11월 말·12월 1일).
- '판다(1)' 비율이 해마다 47~56% 로 한쪽으로 쏠리지 않았다.

**확인 못한 것**
- 실제 데이터로 모델을 학습해 본 성적 (이 노트북에서는 학습하지 않음 — 승인·실험기록 규칙 줄 먼저).
- 실제 데이터에서 걸리는 시간. 2019 시험 해의 학습 줄이 시장마다 100만 줄 넘게 있어서
  랜덤포레스트·LSTM·1D CNN 은 오래 걸릴 수 있다 (DL 은 기본으로 20만 줄만 뽑아 학습).
- 팀원 컴퓨터에서 같은 ✓ 가 나오는지 (각자 `--check` 를 돌려서 확인).

**이상한 것**
- 처음 점검에서 'CSV 매수가' 하나가 ✗ 였다. 원인: CSV 를 읽을 때 소수 마지막 자리(2.2e-16)만 달라짐.
  CSV 를 만든 쪽과 같은 방법(`float_precision="round_trip"`)으로 읽게 고친 뒤 ✓. 매수가 자체는 맞았다.